In [0]:
%sql
-- Dynamic Period Slicer (Fiscal Oct–Sep, Calendar Jan–Dec)
-- Links to dim date by day (DateSk). No dependency on pre-existing flags in dim date.

CREATE OR REPLACE VIEW sts_prd.g_ipd.g_periodslicer_analytics_ipd AS
WITH p AS (
  SELECT
    current_date() AS today,
    year(current_date())  AS ty,                           -- calendar: current year
    month(current_date()) AS tm,                           -- calendar: current month (1-12)
    CASE WHEN month(current_date()) >= 10                  -- fiscal: current FY (Oct–Sep)
         THEN year(current_date()) + 1 ELSE year(current_date()) END AS cfy,
    CASE WHEN month(current_date()) >= 10                  -- fiscal: current FM (1..12)
         THEN month(current_date()) - 9 ELSE month(current_date()) + 3 END AS cfm
),
d0 AS (
  SELECT
    /* derive day key to avoid dependency on a specific column name in dim */
    CAST(date_format(`CalendarDate`, 'yyyyMMdd') AS INT) AS date_sk,
    `CalendarDate` AS d
  FROM sts_prd.s_ipd.s_dimdate_analytics_ipd
),
dd AS (
  SELECT
    date_sk,
    d,
    year(d)  AS y_cal,
    month(d) AS m_cal,
    -- fiscal year/month from calendar date (Oct–Sep)
    CASE WHEN month(d) >= 10 THEN year(d) + 1 ELSE year(d) END          AS fy,
    CASE WHEN month(d) >= 10 THEN month(d) - 9 ELSE month(d) + 3 END    AS fm
  FROM d0
),
x AS (
  SELECT
    dd.*,
    p.ty, p.tm, p.cfy, p.cfm,

    /* Calendar offsets & closed flags */
    CAST(FLOOR(months_between(date_trunc('month', dd.d), date_trunc('month', p.today))) AS INT) AS cal_month_offset,
    (dd.y_cal - p.ty) AS cal_year_offset,
    (dd.y_cal = p.ty AND dd.m_cal <= p.tm) AS cal_ytd_closed,
    (
      CASE WHEN dd.m_cal BETWEEN 1 AND 3 THEN 1
           WHEN dd.m_cal BETWEEN 4 AND 6 THEN 2
           WHEN dd.m_cal BETWEEN 7 AND 9 THEN 3 ELSE 4 END
      =
      CASE WHEN p.tm BETWEEN 1 AND 3 THEN 1
           WHEN p.tm BETWEEN 4 AND 6 THEN 2
           WHEN p.tm BETWEEN 7 AND 9 THEN 3 ELSE 4 END
    ) AND dd.y_cal = p.ty AS cal_qtd_closed,

    /* Fiscal offsets (month index) & closed flags */
    ((dd.fy * 12 + dd.fm) - (p.cfy * 12 + p.cfm)) AS fis_month_offset,
    (dd.fy - p.cfy)                               AS fis_year_offset,
    (dd.fy = p.cfy AND dd.fm <= p.cfm)            AS fis_ytd_closed,
    (
      CASE WHEN dd.fm BETWEEN 1 AND 3 THEN 1
           WHEN dd.fm BETWEEN 4 AND 6 THEN 2
           WHEN dd.fm BETWEEN 7 AND 9 THEN 3 ELSE 4 END
      =
      CASE WHEN p.cfm BETWEEN 1 AND 3 THEN 1
           WHEN p.cfm BETWEEN 4 AND 6 THEN 2
           WHEN p.cfm BETWEEN 7 AND 9 THEN 3 ELSE 4 END
    ) AND dd.fy = p.cfy AS fis_qtd_closed
  FROM dd CROSS JOIN p
),

/* ============================ FISCAL ============================ */
f_ytd_closed      AS (SELECT 'Fiscal'   AS PeriodGroup, 'YTD Fiscal (closed period)'     AS PeriodLabel, 110 AS SortOrder, date_sk AS DateSk FROM x WHERE fis_ytd_closed),
f_qtd_closed      AS (SELECT 'Fiscal',  'Quarter Fiscal (closed period)',                120,             date_sk            FROM x WHERE fis_qtd_closed),
f_month_closed    AS (SELECT 'Fiscal',  'Month Fiscal (closed period)',                  130,             date_sk            FROM x WHERE fis_month_offset = -1),
f_current_year    AS (SELECT 'Fiscal',  'Current Fiscal Year',                           140,             date_sk            FROM x WHERE fy = cfy),
f_current_month   AS (SELECT 'Fiscal',  'Current Fiscal Month',                          150,             date_sk            FROM x WHERE fis_month_offset = 0),
f_prior_ytd       AS (SELECT 'Fiscal',  'Prior YTD Fiscal (closed period)',              160,             date_sk            FROM x WHERE fy = cfy - 1 AND fm <= cfm),
f_prior_year      AS (SELECT 'Fiscal',  'Prior Fiscal Year',                             170,             date_sk            FROM x WHERE fis_year_offset = -1),
f_py_month_closed AS (SELECT 'Fiscal',  'PY Month Fiscal (closed period)',               180,             date_sk            FROM x WHERE fis_month_offset = -13),
f_py_qtd_closed   AS (SELECT 'Fiscal',  'PY Quarter Fiscal (closed period)',             190,             date_sk            FROM x WHERE fy = cfy - 1 AND
                                                                                                                    (CASE WHEN fm BETWEEN 1 AND 3 THEN 1 WHEN fm BETWEEN 4 AND 6 THEN 2 WHEN fm BETWEEN 7 AND 9 THEN 3 ELSE 4 END)
                                                                                                                     =
                                                                                                                    (CASE WHEN cfm BETWEEN 1 AND 3 THEN 1 WHEN cfm BETWEEN 4 AND 6 THEN 2 WHEN cfm BETWEEN 7 AND 9 THEN 3 ELSE 4 END)),
f_ytg             AS (SELECT 'Fiscal',  'YTG Fiscal',                                    200,             date_sk            FROM x WHERE fis_year_offset = 0 AND fis_month_offset >= 1),
f_ytg_closed      AS (SELECT 'Fiscal',  'YTG Fiscal (closed period)',                    210,             date_sk            FROM x WHERE fy = cfy AND NOT fis_ytd_closed),

/* =========================== CALENDAR =========================== */
c_ytd_closed      AS (SELECT 'Calendar','YTD Calendar (closed period)',                  310,             date_sk            FROM x WHERE cal_ytd_closed),
c_qtd_closed      AS (SELECT 'Calendar','Quarter Calendar (closed period)',              320,             date_sk            FROM x WHERE cal_qtd_closed),
c_month_closed    AS (SELECT 'Calendar','Month Calendar (closed period)',                330,             date_sk            FROM x WHERE cal_month_offset = -1),
c_current_year    AS (SELECT 'Calendar','Current Calendar Year',                         340,             date_sk            FROM x WHERE cal_year_offset = 0),
c_current_month   AS (SELECT 'Calendar','Current Calendar Month',                        350,             date_sk            FROM x WHERE cal_month_offset = 0),
c_prior_ytd       AS (SELECT 'Calendar','Prior YTD Calendar (closed period)',            360,             date_sk            FROM x WHERE y_cal = ty - 1 AND m_cal <= tm),
c_prior_year      AS (SELECT 'Calendar','Prior Calendar Year',                           370,             date_sk            FROM x WHERE cal_year_offset = -1),
c_py_month_closed AS (SELECT 'Calendar','PY Month Calendar (closed period)',             380,             date_sk            FROM x WHERE cal_month_offset = -13),
c_ytg             AS (SELECT 'Calendar','YTG Calendar',                                  390,             date_sk            FROM x WHERE cal_year_offset = 0 AND cal_month_offset >= 1),
c_ytg_closed      AS (SELECT 'Calendar','YTG Calendar (closed period)',                  400,             date_sk            FROM x WHERE cal_year_offset = 0 AND NOT cal_ytd_closed),

/* ============================ GENERAL =========================== */
all_periods       AS (SELECT 'General', 'All Periods',                                    999,             date_sk            FROM x)

SELECT * FROM f_ytd_closed
UNION ALL SELECT * FROM f_qtd_closed
UNION ALL SELECT * FROM f_month_closed
UNION ALL SELECT * FROM f_current_year
UNION ALL SELECT * FROM f_current_month
UNION ALL SELECT * FROM f_prior_ytd
UNION ALL SELECT * FROM f_prior_year
UNION ALL SELECT * FROM f_py_month_closed
UNION ALL SELECT * FROM f_py_qtd_closed
UNION ALL SELECT * FROM f_ytg
UNION ALL SELECT * FROM f_ytg_closed
UNION ALL SELECT * FROM c_ytd_closed
UNION ALL SELECT * FROM c_qtd_closed
UNION ALL SELECT * FROM c_month_closed
UNION ALL SELECT * FROM c_current_year
UNION ALL SELECT * FROM c_current_month
UNION ALL SELECT * FROM c_prior_ytd
UNION ALL SELECT * FROM c_prior_year
UNION ALL SELECT * FROM c_py_month_closed
UNION ALL SELECT * FROM c_ytg
UNION ALL SELECT * FROM c_ytg_closed
UNION ALL SELECT * FROM all_periods;
